# Exploring the Dataset: Building the HTTP Access Events Table

**Goal:** Understand how we go from a raw Apache access log annotation file to a relational database table.

This notebook walks through:
1. Loading `intranet_smith_russellmitchell_com-access_log.2` (the raw JSON Lines file containing annotated HTTP access event records)
2. Examining what a single record looks like
3. Transforming the data into a Pandas DataFrame
4. Mapping it to our planned `http_access_events` database table schema

---

**Dataset:** AIT Log Data Set V2.0 — russellmitchell testbed  
**Source:** https://zenodo.org/records/5789064

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.

**Default:** Uses the absolute path to the dataset on this machine:
```
C:/Users/ishaanshetty/DATA-201/russellmitchell/
```

If you're running this on a different machine, update `DATASET_ROOT` to point to your local copy of the `russellmitchell/` folder.

In [1]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path(r"C:/Users/ishaanshetty/DATA-201/russellmitchell")

# The apache access log lives here within the dataset:
# logs/intranet_server/apache2/intranet.smith.russellmitchell.com-access_log.2

# Verify the path exists
if not DATASET_ROOT.exists():
    print(f"ERROR: Dataset not found at: {DATASET_ROOT.resolve()}")
    print("")
    print("Expected directory structure:")
    print("  <workspace>/data-201-security-log-analysis/notebooks/  <-- you are here")
    print("  <workspace>/russellmitchell/                          <-- dataset should be here")
    print("")
    print("Fix: Update DATASET_ROOT above to point to your russellmitchell/ folder.")
else:
    print(f"Dataset found at: {DATASET_ROOT.resolve()}")

Dataset found at: C:\Users\ishaanshetty\DATA-201\russellmitchell


## 1. Load the Raw Apache Access Log File

The file `intranet_smith_russellmitchell_com-access_log.2` is derived from `logs/intranet_server/apache2/intranet.smith.russellmitchell.com-access_log.2` in the russellmitchell dataset.  
Unlike the error log, the **access log** records every HTTP request made to the intranet server — including attacker reconnaissance, directory brute-forcing, web shell uploads, and privilege escalation attempts.  
It contains annotated records in JSON Lines format — each line is one HTTP access event.  
This is the source for our **`http_access_events`** database table.

In [ ]:
import base64
import re
import urllib.parse

import pandas as pd

annotation_path = DATASET_ROOT / 'labels' / 'intranet_server' / 'logs' / 'apache2' / 'intranet.smith.russellmitchell.com-access_log.2'
raw_log_path    = DATASET_ROOT / 'gather' / 'intranet_server' / 'logs' / 'apache2' / 'intranet.smith.russellmitchell.com-access_log.2'

# Load annotation (JSON Lines)
df = pd.read_json(annotation_path, lines=True)
print(f'Loaded {len(df)} annotation records')
print(f'\nColumns: {list(df.columns)}')
print('\nUnique label categories found:')
all_labels = sorted({lbl for labels in df['labels'] for lbl in labels})
for i, label in enumerate(all_labels, 1):
    print(f'  {i:2d}. {label}')

# Apache Combined Log Format
normal_pattern = re.compile(
    r'^(\S+)\s+\S+\s+\S+\s+\[([^\]]+)\]\s+"(\S+)\s+(\S+)\s*(\S*)"\s+(\d{3})\s+(\S+)'
)
# Fallback: null request lines where client connected but sent no data (HTTP 408)
null_pattern = re.compile(
    r'^(\S+)\s+\S+\s+\S+\s+\[([^\]]+)\]\s+"-"\s+(\d{3})\s+(\S+)'
)

def decode_wp_meta(url):
    """Decode base64-encoded shell commands passed via the webshell's wp_meta parameter."""
    m = re.search(r'[?&]wp_meta=([^&\s"]+)', url)
    if not m:
        return None
    try:
        padded = urllib.parse.unquote(m.group(1))
        padded += '=' * (-len(padded) % 4)
        return base64.b64decode(padded).decode('utf-8')
    except Exception:
        return None

raw_rows = []
with open(raw_log_path) as f:
    for lineno, line in enumerate(f, start=1):
        line = line.strip()
        m = normal_pattern.match(line)
        if m:
            ip, ts, method, url, protocol, status, byt = m.groups()
            raw_rows.append({
                'line': lineno, 'client_ip': ip, 'timestamp_raw': ts,
                'method': method, 'url': url, 'protocol': protocol or None,
                'url_path': url.split('?')[0],
                'query_string': url.split('?', 1)[1] if '?' in url else None,
                'status_code': int(status),
                'bytes_sent': None if byt == '-' else int(byt),
                'decoded_command': decode_wp_meta(url),
                'request_type': 'normal'
            })
        else:
            # Null request: client connected but never sent a request line (HTTP 408)
            m2 = null_pattern.match(line)
            if m2:
                ip, ts, status, byt = m2.groups()
                raw_rows.append({
                    'line': lineno, 'client_ip': ip, 'timestamp_raw': ts,
                    'method': None, 'url': None, 'protocol': None,
                    'url_path': None, 'query_string': None,
                    'status_code': int(status),
                    'bytes_sent': None if byt == '-' else int(byt),
                    'decoded_command': None,
                    'request_type': 'null_request'
                })

df_raw = pd.DataFrame(raw_rows)
df_raw['event_timestamp'] = pd.to_datetime(df_raw['timestamp_raw'],
    format='%d/%b/%Y:%H:%M:%S %z', errors='coerce')

# Join annotation with parsed log fields on line number
df_merged = df_raw.merge(df[['line', 'labels', 'rules']], on='line', how='inner')
print(f'\nJoined {len(df_merged)} annotated log records with parsed fields')
print(f'Null requests captured: {len(df_raw[df_raw["request_type"]=="null_request"])}')
print(f'Records with decoded webshell commands: {df_merged["decoded_command"].notna().sum()}')


Loaded 7695 annotation records

Columns: ['line', 'labels', 'rules']

Unique label categories found:
   1. attacker_http
   2. dirb
   3. escalate
   4. foothold
   5. service_scan
   6. webshell_cmd
   7. webshell_upload
   8. wpscan

Joined 7695 annotated log records with parsed fields


## 2. Examine a Single Record

Let's look at what data exists for one record. We'll pick the first entry to see the full raw structure before we flatten anything.

In [2]:
import json

example_record = df.iloc[0].to_dict()

print("Raw data for record 0:\n")
print(json.dumps(example_record, indent=2))

Raw data for record 0:

{
  "line": 832,
  "labels": [
    "service_scan",
    "foothold"
  ],
  "rules": {
    "service_scan": [
      "attacker.service_scan"
    ],
    "foothold": [
      "attacker.service_scan"
    ]
  }
}


### What do these fields mean?

| Field | What It Is | Example |
|-------|-----------|--------|
| `line` | The original line number from the Apache access log file | `832` |
| `labels` | List of attack-phase tags that apply to this log line | `[service_scan, foothold]` |
| `rules` | Dictionary mapping each label to the specific detection rules that matched | `{"service_scan": ["attacker.service_scan"]}` |

The `labels` field tells us **what kind of attack activity** this log line represents.  
The `rules` field tells us **exactly which detection signature** fired for each label.  

Compared to the error log (which had 4 label categories), the access log has **8 label categories**, reflecting the fuller picture of attacker behaviour captured in HTTP access logs:

| Label | What It Represents |
|-------|--------------------|
| `attacker_http` | General attacker HTTP activity on the server |
| `dirb` | Directory brute-force scanning (tool: dirb) |
| `escalate` | Privilege escalation attempts |
| `foothold` | Initial foothold establishment on the target |
| `service_scan` | Service/port scanning activity |
| `webshell_cmd` | Command execution via an uploaded web shell |
| `webshell_upload` | Upload of a malicious web shell to the server |
| `wpscan` | WordPress vulnerability scanning (tool: wpscan) |

Not all records will carry all eight labels — a record tagged only with `wpscan` will only have `wpscan` in its `rules` dict.

## 3. Understand the Label Distribution

The `labels` field categorizes each log line across 8 attack stages. Here we count occurrences using the fully parsed and joined dataset.

In [4]:
from pandasql import sqldf

df_labels = df_merged.explode('labels').rename(columns={'labels': 'label'})

sqldf("SELECT label, COUNT(*) as total_records FROM df_labels GROUP BY label ORDER BY total_records DESC")

,label,total_records
0,foothold,7691
1,attacker_http,7687
2,dirb,4462
3,wpscan,3186
4,webshell_cmd,32
5,service_scan,12
6,escalate,4
7,webshell_upload,3


## 4. Find Records by Label

Filter the parsed log table to isolate specific attack stages. Each query returns real HTTP fields — IP, method, URL, status code — alongside attack labels.

In [5]:
# Web shell upload records — shows exactly which paths the attacker uploaded to
df_merged[df_merged['labels'].apply(lambda x: 'webshell_upload' in x)]\
    [['event_timestamp', 'client_ip', 'method', 'url', 'status_code', 'labels']].head(10)

,event_timestamp,client_ip,method,url_path,query_string,status_code,labels
0,2022-01-24 03:58:20+00:00,172.19.131.174,GET,/,NaN,200,"[attacker_http, foothold, webshell_upload]"
1,2022-01-24 03:58:20+00:00,172.19.131.174,GET,/,p=5,200,"[attacker_http, foothold, webshell_upload]"
2,2022-01-24 03:58:20+00:00,172.19.131.174,POST,/wp-admin/admin-ajax.php,NaN,200,"[attacker_http, foothold, webshell_upload]"


In [6]:
# Web shell command execution records — commands run through the uploaded shell
df_merged[df_merged['labels'].apply(lambda x: 'webshell_cmd' in x)]\
    [['event_timestamp', 'client_ip', 'method', 'url', 'status_code', 'labels']].head(10)

,event_timestamp,client_ip,method,url_path,decoded_command,status_code,labels
0,2022-01-24 03:58:23+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""whoami""]",200,"[attacker_http, foothold, webshell_cmd]"
1,2022-01-24 03:58:25+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""uname"", ""-r""]",200,"[attacker_http, foothold, webshell_cmd]"
2,2022-01-24 03:58:27+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""cat"", ""/etc/profile""]",200,"[attacker_http, foothold, webshell_cmd]"
3,2022-01-24 03:58:28+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""who""]",200,"[attacker_http, foothold, webshell_cmd]"
4,2022-01-24 03:58:30+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""cat"", ""/proc/meminfo""]",200,"[attacker_http, foothold, webshell_cmd]"
5,2022-01-24 03:58:33+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""uname"", ""-a""]",200,"[attacker_http, foothold, webshell_cmd]"
6,2022-01-24 03:58:36+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""id""]",200,"[attacker_http, foothold, webshell_cmd]"
7,2022-01-24 03:58:39+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""df"", ""-h""]",200,"[attacker_http, foothold, webshell_cmd]"
8,2022-01-24 03:58:40+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""netstat"", ""-nat""]",200,"[attacker_http, foothold, webshell_cmd]"
9,2022-01-24 03:58:42+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""id""]",200,"[attacker_http, foothold, webshell_cmd]"


In [3]:
# Privilege escalation records
df_merged[df_merged['labels'].apply(lambda x: 'escalate' in x)]\
    [['event_timestamp', 'client_ip', 'method', 'url', 'status_code', 'labels']].head(10)

,event_timestamp,client_ip,method,url_path,query_string,status_code,labels
0,2022-01-24 03:56:53+00:00,172.19.131.174,GET,/,NaN,200,"[service_scan, foothold]"
1,2022-01-24 03:56:59+00:00,172.19.131.174,GET,/,NaN,200,"[service_scan, foothold]"
2,2022-01-24 03:57:01+00:00,172.19.131.174,POST,/sdk,NaN,404,"[attacker_http, foothold, service_scan]"
3,2022-01-24 03:57:01+00:00,172.19.131.174,GET,/nmaplowercheck1642996621,NaN,404,"[attacker_http, foothold, service_scan]"
4,2022-01-24 03:57:01+00:00,172.19.131.174,GET,/nmaplowercheck1642996621,NaN,404,"[attacker_http, foothold, service_scan]"
5,2022-01-24 03:57:01+00:00,172.19.131.174,POST,/sdk,NaN,404,"[attacker_http, foothold, service_scan]"
6,2022-01-24 03:57:01+00:00,172.19.131.174,GET,/,NaN,200,"[attacker_http, foothold, service_scan]"
7,2022-01-24 03:57:01+00:00,172.19.131.174,GET,/,NaN,200,"[attacker_http, foothold, service_scan]"
8,2022-01-24 03:57:01+00:00,172.19.131.174,GET,/HNAP1,NaN,404,"[attacker_http, foothold, service_scan]"
9,2022-01-24 03:57:01+00:00,172.19.131.174,GET,/HNAP1,NaN,404,"[attacker_http, foothold, service_scan]"


In [ ]:
# Full web shell attack chain: upload -> command execution -> escalation in chronological order
df_merged[
    df_merged['labels'].apply(lambda x: any(lbl in x for lbl in ['webshell_upload', 'webshell_cmd', 'escalate']))
][['event_timestamp', 'client_ip', 'method', 'url', 'status_code', 'labels']].sort_values('event_timestamp')

,event_timestamp,client_ip,method,url_path,decoded_command,status_code,labels
0,2022-01-24 03:58:20+00:00,172.19.131.174,GET,/,NaN,200,"[attacker_http, foothold, webshell_upload]"
1,2022-01-24 03:58:20+00:00,172.19.131.174,GET,/,NaN,200,"[attacker_http, foothold, webshell_upload]"
2,2022-01-24 03:58:20+00:00,172.19.131.174,POST,/wp-admin/admin-ajax.php,NaN,200,"[attacker_http, foothold, webshell_upload]"
3,2022-01-24 03:58:23+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""whoami""]",200,"[attacker_http, foothold, webshell_cmd]"
4,2022-01-24 03:58:25+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""uname"", ""-r""]",200,"[attacker_http, foothold, webshell_cmd]"
5,2022-01-24 03:58:27+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""cat"", ""/etc/profile""]",200,"[attacker_http, foothold, webshell_cmd]"
6,2022-01-24 03:58:28+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""who""]",200,"[attacker_http, foothold, webshell_cmd]"
7,2022-01-24 03:58:30+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""cat"", ""/proc/meminfo""]",200,"[attacker_http, foothold, webshell_cmd]"
8,2022-01-24 03:58:33+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""uname"", ""-a""]",200,"[attacker_http, foothold, webshell_cmd]"
9,2022-01-24 03:58:36+00:00,172.19.131.174,GET,/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php,"[""id""]",200,"[attacker_http, foothold, webshell_cmd]"


In [9]:
# dirb directory brute-force records — high volume of 404s from the same IP
df_merged[df_merged['labels'].apply(lambda x: 'dirb' in x)]\
    [['event_timestamp', 'client_ip', 'method', 'url', 'status_code', 'labels']].head(10)

,event_timestamp,client_ip,method,url_path,query_string,status_code,labels
0,2022-01-24 03:57:26+00:00,172.19.131.174,GET,/randomfile1,NaN,404,"[attacker_http, foothold, dirb]"
1,2022-01-24 03:57:26+00:00,172.19.131.174,GET,/frand2,NaN,404,"[attacker_http, foothold, dirb]"
2,2022-01-24 03:57:26+00:00,172.19.131.174,GET,/.bash_history,NaN,404,"[attacker_http, foothold, dirb]"
3,2022-01-24 03:57:26+00:00,172.19.131.174,GET,/.bashrc,NaN,404,"[attacker_http, foothold, dirb]"
4,2022-01-24 03:57:26+00:00,172.19.131.174,GET,/.cache,NaN,404,"[attacker_http, foothold, dirb]"
5,2022-01-24 03:57:26+00:00,172.19.131.174,GET,/.config,NaN,404,"[attacker_http, foothold, dirb]"
6,2022-01-24 03:57:26+00:00,172.19.131.174,GET,/.cvs,NaN,404,"[attacker_http, foothold, dirb]"
7,2022-01-24 03:57:26+00:00,172.19.131.174,GET,/.cvsignore,NaN,404,"[attacker_http, foothold, dirb]"
8,2022-01-24 03:57:26+00:00,172.19.131.174,GET,/.forward,NaN,404,"[attacker_http, foothold, dirb]"
9,2022-01-24 03:57:26+00:00,172.19.131.174,GET,/.git/HEAD,NaN,404,"[attacker_http, foothold, dirb]"


In [10]:
# wpscan WordPress vulnerability scanning records
df_merged[df_merged['labels'].apply(lambda x: 'wpscan' in x)]\
    [['event_timestamp', 'client_ip', 'method', 'url', 'status_code', 'labels']].head(10)

,event_timestamp,client_ip,method,url_path,query_string,status_code,labels
0,2022-01-24 03:57:52+00:00,172.19.131.174,GET,/,NaN,200,"[attacker_http, foothold, wpscan]"
1,2022-01-24 03:57:52+00:00,172.19.131.174,GET,/,NaN,200,"[attacker_http, foothold, wpscan]"
2,2022-01-24 03:57:52+00:00,172.19.131.174,HEAD,/,NaN,200,"[attacker_http, foothold, wpscan]"
3,2022-01-24 03:57:52+00:00,172.19.131.174,GET,/7f21b6c.html,NaN,404,"[attacker_http, foothold, wpscan]"
4,2022-01-24 03:57:53+00:00,172.19.131.174,HEAD,/robots.txt,NaN,404,"[attacker_http, foothold, wpscan]"
5,2022-01-24 03:57:53+00:00,172.19.131.174,HEAD,/fantastico_fileslist.txt,NaN,404,"[attacker_http, foothold, wpscan]"
6,2022-01-24 03:57:53+00:00,172.19.131.174,HEAD,/searchreplacedb2.php,NaN,404,"[attacker_http, foothold, wpscan]"
7,2022-01-24 03:57:53+00:00,172.19.131.174,POST,/xmlrpc.php,NaN,200,"[attacker_http, foothold, wpscan]"
8,2022-01-24 03:57:53+00:00,172.19.131.174,HEAD,/readme.html,NaN,200,"[attacker_http, foothold, wpscan]"
9,2022-01-24 03:57:53+00:00,172.19.131.174,GET,/readme.html,NaN,200,"[attacker_http, foothold, wpscan]"


## 5. Records Carrying Only One Label

Lines that triggered exactly one detection rule — useful for validating individual signatures in isolation.

In [11]:
df_single = df_merged[df_merged['labels'].apply(len) == 1].copy()
df_single['label'] = df_single['labels'].apply(lambda x: x[0])

df_single.groupby('label').size().reset_index(name='total').sort_values('total', ascending=False)

,label,total


In [12]:
# Show the actual log entries for single-label records
df_single[['event_timestamp', 'client_ip', 'method', 'url', 'status_code', 'label']].head(10)

,event_timestamp,client_ip,method,url_path,status_code,label


## 6. What Signatures Does Each Label Fire?

The `rules` field contains a nested dictionary of signature matches. Flatten it to document all sub-fields and how they map to `http.request.sig_match` in our schema.

In [13]:
sig_rows = []
for _, row in df_merged.iterrows():
    if isinstance(row['rules'], dict):
        for label, sigs in row['rules'].items():
            for sig in sigs:
                sig_rows.append({'label': label, 'signature': sig})

df_sigs = pd.DataFrame(sig_rows).drop_duplicates()
df_sigs.sort_values('label')

,label,signature
0,attacker_http,attacker.foothold.apache.access
1,attacker_http,attacker.foothold.apache.access_error
2,dirb,attacker.dirb.time
3,escalate,attacker.escalate.webshell.cmd.http_prepare_crack
4,foothold,attacker.service_scan
5,foothold,attacker.foothold.apache.access
6,foothold,attacker.foothold.apache.access_error
7,service_scan,attacker.service_scan
8,webshell_cmd,attacker.escalate.webshell.cmd.http
9,webshell_cmd,attacker.escalate.webshell.cmd.http_prepare_crack


## 7. Mapping to the Database Schema

Here's how this data maps to our planned **`http_access_events`** table in PostgreSQL:

| Raw Field | DB Column | SQL Type | Notes |
|-----------|-----------|----------|-------|
| *(auto-generated)* | `http_access_event_id` | `SERIAL PRIMARY KEY` | Auto-incrementing ID |
| `line` | `event_id` | `INTEGER NOT NULL` | Original line number from the log file |
| `event_timestamp` | `event_timestamp` | `TIMESTAMP WITH TIME ZONE` | Parsed from raw log |
| `client_ip` | `client_ip` | `INET` | Attacker IP address |
| `method` | `http_method` | `VARCHAR(10)` | e.g. `GET`, `POST` |
| `url` | `request_url` | `TEXT` | Requested path |
| `status_code` | `status_code` | `SMALLINT` | HTTP response code |
| `bytes_sent` | `bytes_sent` | `INTEGER` | Response size in bytes |
| `labels` | `http_access_event_category` | `TEXT[]` | Array of attack-phase labels |
| `rules` | `http_access_signature_matches` | `JSONB` | Full nested signature match dictionary |
| *(auto-generated)* | `created_at` | `TIMESTAMP DEFAULT CURRENT_TIMESTAMP` | Added at insert time |

### The SQL `CREATE TABLE` statement:

```sql
CREATE TABLE http_access_events (
    http_access_event_id          SERIAL PRIMARY KEY,
    event_id                      INTEGER NOT NULL,
    event_timestamp               TIMESTAMP WITH TIME ZONE,
    client_ip                     INET,
    http_method                   VARCHAR(10),
    request_url                   TEXT,
    status_code                   SMALLINT,
    bytes_sent                    INTEGER,
    http_access_event_category    TEXT[],
    http_access_signature_matches JSONB,
    created_at                    TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
```

In [14]:
# Preview of the http_access_events table as it will look in PostgreSQL
db_preview = df_merged[['line', 'event_timestamp', 'client_ip', 'method', 'url', 'status_code', 'bytes_sent', 'labels', 'rules']].copy()
db_preview.columns = ['event_id', 'event_timestamp', 'client_ip', 'http_method', 'request_url', 'status_code', 'bytes_sent', 'http_access_event_category', 'http_access_signature_matches']
db_preview.head(10)

,http_access_event_id,event_id,event_timestamp,client_ip,http_method,request_url,query_string,status_code,bytes_sent,decoded_command,http_access_event_category,http_access_signature_matches
0,1,832,2022-01-24 03:56:53+00:00,172.19.131.174,GET,/,NaN,200,17051,NaN,"[service_scan, foothold]","{'service_scan': ['attacker.service_scan'], 'foothold': ['attacker.service_scan']}"
1,2,835,2022-01-24 03:56:59+00:00,172.19.131.174,GET,/,NaN,200,21164,NaN,"[service_scan, foothold]","{'service_scan': ['attacker.service_scan'], 'foothold': ['attacker.service_scan']}"
2,3,836,2022-01-24 03:57:01+00:00,172.19.131.174,POST,/sdk,NaN,404,360,NaN,"[attacker_http, foothold, service_scan]","{'attacker_http': ['attacker.foothold.apache.access'], 'foothold': ['attacker.foothold.apache.access', 'attacker.service_scan'], 'service_scan': ['attacker.service_scan']}"
3,4,837,2022-01-24 03:57:01+00:00,172.19.131.174,GET,/nmaplowercheck1642996621,NaN,404,360,NaN,"[attacker_http, foothold, service_scan]","{'attacker_http': ['attacker.foothold.apache.access'], 'foothold': ['attacker.foothold.apache.access', 'attacker.service_scan'], 'service_scan': ['attacker.service_scan']}"
4,5,838,2022-01-24 03:57:01+00:00,172.19.131.174,GET,/nmaplowercheck1642996621,NaN,404,3287,NaN,"[attacker_http, foothold, service_scan]","{'attacker_http': ['attacker.foothold.apache.access'], 'foothold': ['attacker.foothold.apache.access', 'attacker.service_scan'], 'service_scan': ['attacker.service_scan']}"
5,6,839,2022-01-24 03:57:01+00:00,172.19.131.174,POST,/sdk,NaN,404,3287,NaN,"[attacker_http, foothold, service_scan]","{'attacker_http': ['attacker.foothold.apache.access'], 'foothold': ['attacker.foothold.apache.access', 'attacker.service_scan'], 'service_scan': ['attacker.service_scan']}"
6,7,840,2022-01-24 03:57:01+00:00,172.19.131.174,GET,/,NaN,200,21164,NaN,"[attacker_http, foothold, service_scan]","{'attacker_http': ['attacker.foothold.apache.access'], 'foothold': ['attacker.foothold.apache.access', 'attacker.service_scan'], 'service_scan': ['attacker.service_scan']}"
7,8,841,2022-01-24 03:57:01+00:00,172.19.131.174,GET,/,NaN,200,18120,NaN,"[attacker_http, foothold, service_scan]","{'attacker_http': ['attacker.foothold.apache.access'], 'foothold': ['attacker.foothold.apache.access', 'attacker.service_scan'], 'service_scan': ['attacker.service_scan']}"
8,9,842,2022-01-24 03:57:01+00:00,172.19.131.174,GET,/HNAP1,NaN,404,360,NaN,"[attacker_http, foothold, service_scan]","{'attacker_http': ['attacker.foothold.apache.access'], 'foothold': ['attacker.foothold.apache.access', 'attacker.service_scan'], 'service_scan': ['attacker.service_scan']}"
9,10,843,2022-01-24 03:57:01+00:00,172.19.131.174,GET,/HNAP1,NaN,404,3287,NaN,"[attacker_http, foothold, service_scan]","{'attacker_http': ['attacker.foothold.apache.access'], 'foothold': ['attacker.foothold.apache.access', 'attacker.service_scan'], 'service_scan': ['attacker.service_scan']}"
